# Taller Interactivo: Errores e Incertidumbre en la Medición
### Física BR · Física Mecánica

La **metrología** —la ciencia de las mediciones— es el punto de partida de toda práctica experimental en ingeniería. Antes de operar instrumentos complejos, el ingeniero debe entender qué significa medir, qué tan confiable es un resultado y cómo cuantificar la incertidumbre.

**Contenido:**
- **Punto 1:** Exactitud vs. precisión
- **Punto 2:** Tipos de error y métricas (absoluto, relativo, porcentual)
- **Punto 3:** Estadística de mediciones (promedio, desviación estándar, error de la media)
- **Punto 4:** Propagación de errores en magnitudes derivadas

Cada punto incluye un esquema, controles interactivos, solución paso a paso, una pregunta de razonamiento (*defiende tu conclusión*) y un recuadro de tutor socrático (TIP-IA). Nota formativa de 0,0 a 5,0.

> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*

---


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass


## Punto 1 — Exactitud vs. precisión  ★  [CL, UCM]
Dos conceptos que suelen confundirse pero son distintos:
- **Exactitud (veracidad):** qué tan cerca está el valor medido del valor verdadero.
- **Precisión:** qué tan reproducibles son las mediciones repetidas entre sí, sin importar su cercanía al valor verdadero.

Un instrumento puede ser **preciso sin ser exacto** (error sistemático: siempre se desvía igual) o **exacto en promedio sin ser preciso** (mucha dispersión aleatoria). La analogía clásica es el tiro al blanco.

> 🔧 **Aplicación en ingeniería:** en manufactura, un instrumento preciso pero no exacto produce piezas consistentemente fuera de tolerancia (todas mal, del mismo modo). Uno exacto pero impreciso da piezas que en promedio están bien, pero individualmente varían. El Instituto Nacional de Metrología (INM) de Colombia calibra instrumentos para garantizar ambas cosas.

> 🤖 **TIP-IA — Pon a prueba tu intuición:**
> Pídele a un asistente: *"Voy a sostener que si un instrumento es muy preciso, entonces es muy exacto. Dame el mejor contraargumento y un ejemplo donde falle."* (Pista: piensa en una báscula descalibrada que siempre marca 2 kg de más). La IA es espejo de tu razonamiento, no sustituto.

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,axs=plt.subplots(1,4,figsize=(12,3.4))
    np.random.seed(3)
    casos=[
        ("Preciso y exacto",0.0,0.0,0.08,'#27ae60'),
        ("Preciso, no exacto",0.5,0.3,0.08,'#e67e22'),
        ("Exacto, no preciso",0.0,0.0,0.35,'#2980b9'),
        ("Ni preciso ni exacto",0.5,0.3,0.35,'#c0392b'),
    ]
    for ax,(titulo,cx,cy,disp,col) in zip(axs,casos):
        # dianas concéntricas
        for r in [1.0,0.66,0.33]:
            ax.add_patch(plt.Circle((0,0),r,fill=False,ec='#95a5a6',lw=1))
        ax.add_patch(plt.Circle((0,0),0.10,color='#e74c3c',alpha=0.5))
        # disparos
        xs=cx+np.random.randn(6)*disp; ys=cy+np.random.randn(6)*disp
        ax.scatter(xs,ys,s=45,color=col,zorder=5,edgecolor='white',linewidth=1)
        ax.set_xlim(-1.2,1.2); ax.set_ylim(-1.2,1.2); ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(titulo,fontsize=9.5,color=col)
    plt.suptitle('Exactitud vs. precisión: la analogía del tiro al blanco',fontsize=12,fontweight='bold',color='#2c3e50')
    plt.tight_layout(); plt.show()
esquema()

def blanco(sesgo=0.0, dispersion=0.15):
    np.random.seed(7)
    fig,ax=plt.subplots(figsize=(5.5,5.5))
    for r in [1.0,0.66,0.33]:
        ax.add_patch(plt.Circle((0,0),r,fill=False,ec='#95a5a6',lw=1.2))
    ax.add_patch(plt.Circle((0,0),0.10,color='#e74c3c',alpha=0.5))
    xs=sesgo+np.random.randn(8)*dispersion; ys=np.random.randn(8)*dispersion
    ax.scatter(xs,ys,s=70,color='#2980b9',zorder=5,edgecolor='white',linewidth=1.2)
    # centro de los disparos (promedio)
    ax.scatter([xs.mean()],[ys.mean()],s=180,marker='+',color='#c0392b',zorder=6,linewidth=2.5)
    exacto='alta' if abs(sesgo)<0.15 else 'baja'
    preciso='alta' if dispersion<0.20 else 'baja'
    ax.set_xlim(-1.3,1.3); ax.set_ylim(-1.3,1.3); ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(f'Exactitud {exacto} · Precisión {preciso}\n(+ rojo = promedio de los disparos)',fontsize=11)
    plt.tight_layout(); plt.show()

interact(blanco,
    sesgo=FloatSlider(value=0.0,min=-0.6,max=0.6,step=0.1,description='sesgo (exactitud)'),
    dispersion=FloatSlider(value=0.15,min=0.05,max=0.45,step=0.05,description='dispersión (precisión)'));

def solucion():
    print("EXACTITUD (veracidad):")
    print("   Qué tan cerca está el valor medido del valor VERDADERO.")
    print("   Se relaciona con el error SISTEMÁTICO (sesgo constante).")
    print("")
    print("PRECISIÓN:")
    print("   Qué tan reproducibles son las mediciones repetidas ENTRE SÍ.")
    print("   Se relaciona con el error ALEATORIO (dispersión).")
    print("")
    print("Los cuatro casos del tiro al blanco:")
    print("   • Preciso y exacto: disparos juntos y en el centro (ideal).")
    print("   • Preciso, no exacto: disparos juntos pero desviados (error sistemático).")
    print("   • Exacto, no preciso: disparos dispersos pero centrados en promedio.")
    print("   • Ni preciso ni exacto: disparos dispersos y desviados.")
    print("")
    print("Clave: promediar muchas medidas mejora la PRECISIÓN del promedio,")
    print("pero NO corrige un error sistemático (la falta de exactitud).")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_opcion_multiple("Un termómetro marca siempre 2°C de más, pero sus lecturas repetidas coinciden. ¿Cómo se describe?",
    ["Exacto pero impreciso","Preciso pero inexacto (error sistemático de +2°C)","Preciso y exacto","Ni preciso ni exacto"],1,
    "Lecturas que coinciden = alta precisión. Pero siempre 2°C de más = sesgo sistemático = baja exactitud.", id_preg="Err_p1_q_1", peso=1.0)
quiz_opcion_multiple("Repites una medición muchas veces y promedias. ¿Qué mejora ese promedio?",
    ["Corrige el error sistemático","Mejora la precisión del resultado (reduce el error aleatorio), pero NO corrige el sesgo sistemático","Aumenta la resolución del instrumento","No sirve de nada"],1,
    "Promediar reduce el efecto del error aleatorio (más precisión), pero un sesgo constante permanece: la exactitud no mejora promediando.", id_preg="Err_p1_q_2", peso=1.0)
quiz_opcion_multiple("Un instrumento da lecturas muy dispersas pero cuyo promedio coincide con el valor real. Es:",
    ["Preciso pero inexacto","Exacto (en promedio) pero impreciso","Preciso y exacto","Inservible"],1,
    "El promedio coincide con el valor real = exacto en promedio. Pero la gran dispersión = baja precisión.", id_preg="Err_p1_q_3", peso=1.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante dice 'mi balanza da siempre el mismo valor, así que es confiable'. ¿Correcto o solo convincente?",
    ["Correcto: repetibilidad = confiabilidad","Solo convincente: dar siempre el mismo valor es PRECISIÓN, pero si está descalibrada será siempre el mismo valor EQUIVOCADO (inexacto). Repetibilidad no garantiza veracidad","Correcto si es digital","Depende de la marca"],1,
    "Una balanza descalibrada es muy precisa (repite el valor) pero inexacta (siempre mal). Confundir precisión con exactitud es el error clásico: la repetibilidad no detecta el sesgo sistemático.", id_preg="Err_p1_concep_4", peso=2.0)


## Punto 2 — Tipos de error y métricas  ★★  [CL, UCM, ULC]
**Tres tipos de error según su origen:**
- **Sistemático:** se repite igual en todas las mediciones (mala calibración, método incorrecto). No se reduce promediando.
- **Aleatorio:** varía impredeciblemente de una medición a otra. Se reduce promediando más mediciones.
- **Burdo:** por descuido evidente (leer mal, anotar mal). Debe identificarse y eliminarse antes del análisis.

**Tres métricas para cuantificar el error:**
- Error absoluto: Δx (en las mismas unidades de la medida)
- Error relativo: ε = Δx/x̄ (adimensional)
- Error porcentual: ε% = (Δx/x̄)×100 %

> 🔧 **Aplicación en ingeniería:** distinguir error sistemático de aleatorio es clave en control de calidad. Un error sistemático en un sensor de presión de un ducto (API/ASME) puede pasar todas las pruebas repetidas y aun así estar mal: la repetibilidad (precisión) no garantiza veracidad (exactitud). El error relativo permite comparar la calidad de mediciones de magnitudes muy distintas.

> 🤖 **TIP-IA — Verificador crítico:**
> Escribe: *"Clasifiqué los errores de mi experimento en sistemáticos y aleatorios. Actúa como evaluador escéptico: pregúntame si cada error que llamé 'aleatorio' realmente se reduce al promediar, y si alguno que llamé 'sistemático' podría en realidad ser un descuido puntual. No me des la respuesta."* 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4))
    np.random.seed(1)
    # IZQUIERDA: sistemático vs aleatorio en el tiempo
    n=20; x=np.arange(n); verdadero=10.0
    sist=verdadero+1.5+np.random.randn(n)*0.15  # sesgo +1.5
    alea=verdadero+np.random.randn(n)*0.8        # sin sesgo, más dispersión
    ax1.axhline(verdadero,color='#27ae60',lw=2,ls='--',label='valor verdadero')
    ax1.scatter(x,sist,color='#e67e22',s=30,label='error sistemático (sesgado)',zorder=4)
    ax1.scatter(x,alea,color='#2980b9',s=30,label='error aleatorio (disperso)',zorder=4)
    ax1.axhline(sist.mean(),color='#e67e22',lw=1,ls=':')
    ax1.set_xlabel('número de medición'); ax1.set_ylabel('valor medido')
    ax1.set_title('Sistemático (sesgo fijo) vs aleatorio (disperso)',fontsize=10)
    ax1.legend(fontsize=7.5,loc='upper right')
    # DERECHA: histogramas
    ax2.hist(alea,bins=8,color='#2980b9',alpha=0.5,label='aleatorio')
    ax2.hist(sist,bins=8,color='#e67e22',alpha=0.5,label='sistemático')
    ax2.axvline(verdadero,color='#27ae60',lw=2,ls='--',label='verdadero')
    ax2.set_xlabel('valor medido'); ax2.set_ylabel('frecuencia')
    ax2.set_title('El sistemático corre el centro; el aleatorio lo ensancha',fontsize=10)
    ax2.legend(fontsize=8)
    plt.tight_layout(); plt.show()
esquema()

def metricas(x_medido=21.4, x_referencia=21.0):
    err_abs=abs(x_medido-x_referencia)
    err_rel=err_abs/x_medido if x_medido!=0 else 0
    err_pct=err_rel*100
    fig,ax=plt.subplots(figsize=(8,3.5))
    ax.barh(['Error\nabsoluto','Error\nporcentual'],[err_abs,err_pct],
            color=['#e67e22','#c0392b'],alpha=0.75)
    ax.text(err_abs,0,f'  {err_abs:.3g}',va='center',fontsize=11,fontweight='bold')
    ax.text(err_pct,1,f'  {err_pct:.3g}%',va='center',fontsize=11,fontweight='bold')
    ax.set_title(f'Medido={x_medido:.3g} · Referencia={x_referencia:.3g}')
    ax.set_xlabel('magnitud del error')
    plt.tight_layout(); plt.show()

interact(metricas,
    x_medido=FloatSlider(value=21.4,min=18,max=24,step=0.2,description='medido'),
    x_referencia=FloatSlider(value=21.0,min=18,max=24,step=0.2,description='referencia'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            return _re.sub(r'(\d)\.(\d)', r'\1,\2', t)
        print(*[_f(x) for x in a], **k)
    _pc("TIPOS DE ERROR:")
    _pc("   • Sistemático: sesgo constante (todas las medidas corridas igual). NO se")
    _pc("     reduce promediando; se corrige recalibrando o cambiando el método.")
    _pc("   • Aleatorio: dispersión impredecible. SÍ se reduce promediando más medidas.")
    _pc("   • Burdo: equivocación puntual (mal leído/anotado). Se descarta antes de analizar.")
    _pc("")
    _pc("MÉTRICAS (ejemplo: medido=21,4; referencia=21,0):")
    ea=abs(21.4-21.0); er=ea/21.4
    _pc(f"   Error absoluto: Δx = |21,4 − 21,0| = {ea:.3g}")
    _pc(f"   Error relativo: ε = Δx/x = {er:.4g}")
    _pc(f"   Error porcentual: ε% = {er*100:.3g}%")
    _pc("   El relativo/porcentual permite comparar mediciones de magnitudes distintas:")
    _pc("   un error de 1 mm es grave en un tornillo, trivial en un puente.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Una medición da 21,4 cm cuando el valor de referencia es 21,0 cm. Calcule el error porcentual.",1.87,0.05,"%",
    "ε%=|21,4−21,0|/21,4×100 = 0,4/21,4×100 ≈ 1,87%.", id_preg="Err_p2_q_5", peso=2.0)
quiz_opcion_multiple("¿Cuál tipo de error se reduce promediando muchas mediciones?",
    ["El sistemático","El aleatorio","El burdo","Ninguno"],1,
    "El error aleatorio se promedia hacia cero con muchas medidas. El sistemático (sesgo fijo) permanece; el burdo se descarta.", id_preg="Err_p2_q_6", peso=2.0)
quiz_opcion_multiple("¿Por qué el error relativo (o porcentual) es útil para comparar mediciones?",
    ["Porque siempre es más pequeño","Porque es adimensional: permite comparar la calidad de mediciones de magnitudes muy distintas","Porque elimina el error","Porque no necesita unidades verdaderas"],1,
    "Al ser adimensional (Δx/x̄), el error relativo compara mediciones de escalas distintas: 1 mm en un tornillo vs. en un puente son errores relativos muy diferentes.", id_preg="Err_p2_q_7", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: alguien afirma 'para mejorar mi medición con error sistemático, tomo más datos y promedio'. ¿Funciona?",
    ["Sí, promediar siempre mejora todo","No: promediar reduce el error ALEATORIO, pero el sistemático es un sesgo fijo que permanece por más que promedies. Hay que recalibrar o cambiar el método","Sí, si tomo más de 30 datos","No, nunca sirve promediar"],1,
    "Promediar ataca el error aleatorio, no el sistemático. Un sesgo constante sobrevive a cualquier cantidad de promedios. Creer que 'más datos' cura todo es la trampa.", id_preg="Err_p2_concep_8", peso=2.0)


## Punto 3 — Estadística de mediciones  ★★  [UCM, PRC, ULC]
Cuando se mide **n veces** la misma magnitud, la estadística resume el resultado:
- **Promedio:** x̄ = (1/n)·Σxᵢ  → la mejor estimación del valor
- **Desviación estándar:** σ = √[Σ(xᵢ−x̄)²/(n−1)]  → cuánto se dispersan las medidas
- **Error estándar de la media:** σ_x̄ = σ/√n  → la incertidumbre del promedio

El resultado se reporta como **x̄ ± σ_x̄**. Nota clave: al aumentar n, el error de la media baja como 1/√n (cuadruplicar las medidas solo reduce a la mitad el error).

> 🔧 **Aplicación en ingeniería:** el control topográfico en obra civil requiere incertidumbres milimétricas; se logran promediando múltiples visadas. Entender que el error de la media baja como 1/√n permite decidir cuántas mediciones valen la pena: pasar de 4 a 16 medidas solo reduce el error a la mitad, con 4 veces el trabajo.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pega la fórmula σ_x̄ = σ/√n a un asistente: *"Explícame qué significa físicamente que el error de la media baje como 1/√n, qué le pasa a la gráfica cuando n crece, y por qué llega un punto en que medir más ya casi no ayuda. No me des números, explícame la física."* 

In [ ]:
from ipywidgets import IntSlider
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4))
    # IZQUIERDA: nube de mediciones con promedio y desviación
    np.random.seed(5)
    datos=21.4+np.random.randn(12)*0.15
    x=np.arange(len(datos))
    prom=datos.mean(); desv=datos.std(ddof=1)
    ax1.scatter(x,datos,color='#2980b9',s=45,zorder=4)
    ax1.axhline(prom,color='#c0392b',lw=2,label=f'promedio={prom:.3g}')
    ax1.axhspan(prom-desv,prom+desv,color='#e67e22',alpha=0.12,label=f'±σ={desv:.3g}')
    ax1.set_xlabel('número de medición'); ax1.set_ylabel('valor medido (cm)')
    ax1.set_title('Mediciones repetidas: promedio ± desviación',fontsize=10)
    ax1.legend(fontsize=8)
    # DERECHA: error de la media decae como 1/√n
    n=np.arange(1,51); err=0.15/np.sqrt(n)
    ax2.plot(n,err,color='#8e44ad',lw=2.5)
    for nn in [4,16]:
        ax2.scatter([nn],[0.15/np.sqrt(nn)],color='#c0392b',s=60,zorder=5)
        ax2.annotate(f'n={nn}',xy=(nn,0.15/np.sqrt(nn)),xytext=(nn+3,0.15/np.sqrt(nn)+0.01),fontsize=8)
    ax2.set_xlabel('número de mediciones n'); ax2.set_ylabel('error de la media σ/√n')
    ax2.set_title('El error de la media baja como 1/√n',fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def estadistica(n=5, dispersion=0.15):
    np.random.seed(42)
    datos=21.4+np.random.randn(n)*dispersion
    prom=datos.mean(); desv=datos.std(ddof=1) if n>1 else 0; err_media=desv/np.sqrt(n) if n>1 else 0
    fig,ax=plt.subplots(figsize=(8,4))
    ax.scatter(np.arange(n),datos,color='#2980b9',s=50,zorder=4,label='mediciones')
    ax.axhline(prom,color='#c0392b',lw=2,label=f'x̄={prom:.3g} cm')
    ax.axhspan(prom-err_media,prom+err_media,color='#27ae60',alpha=0.2,label=f'±σ_x̄={err_media:.3g}')
    ax.set_xlabel('número de medición'); ax.set_ylabel('valor (cm)')
    ax.set_title(f'n={n}: L=({prom:.2f} ± {err_media:.2f}) cm  ·  σ={desv:.3g} cm')
    ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

interact(estadistica,
    n=IntSlider(value=5,min=2,max=30,step=1,description='n mediciones'),
    dispersion=FloatSlider(value=0.15,min=0.05,max=0.4,step=0.05,description='dispersión'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            return _re.sub(r'(\d)\.(\d)', r'\1,\2', t)
        print(*[_f(x) for x in a], **k)
    datos=np.array([21.4,21.5,21.3,21.6,21.4])
    prom=datos.mean(); desv=datos.std(ddof=1); em=desv/np.sqrt(len(datos))
    _pc("Ejemplo: 5 mediciones = [21,4; 21,5; 21,3; 21,6; 21,4] cm")
    _pc(f"   Promedio: x̄ = (Σxᵢ)/5 = {prom:.3g} cm")
    _pc(f"   Desviación estándar: σ = √[Σ(xᵢ−x̄)²/(n−1)] = {desv:.3g} cm")
    _pc(f"   Error estándar de la media: σ_x̄ = σ/√n = {desv:.3g}/√5 = {em:.3g} cm")
    _pc(f"   Resultado reportado: L = ({prom:.1f} ± {em:.1f}) cm")
    _pc("")
    _pc("Por qué (n−1) y no n: al usar el propio promedio de los datos, se 'gasta'")
    _pc("un grado de libertad; dividir por (n−1) corrige el sesgo (corrección de Bessel).")
    _pc("El error de la media baja como 1/√n: cuadruplicar n solo lo reduce a la mitad.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Cinco mediciones: 21,4; 21,5; 21,3; 21,6; 21,4 cm. Calcule el promedio (en cm).",21.4,0.02,"cm",
    "x̄=(21,4+21,5+21,3+21,6+21,4)/5=21,44≈21,4 cm.", id_preg="Err_p3_q_9", peso=2.0)
quiz_numerico("Para esas cinco mediciones, la desviación estándar (n−1) es aprox. (en cm):",0.114,0.15,"cm",
    "σ=√[Σ(xᵢ−x̄)²/(n−1)]≈0,114 cm.", id_preg="Err_p3_q_10", peso=2.0)
quiz_opcion_multiple("¿Por qué se divide por (n−1) y no por n en la desviación estándar muestral?",
    ["Por convención arbitraria","Porque al usar el promedio de los propios datos se pierde un grado de libertad; (n−1) corrige el sesgo (corrección de Bessel)","Porque n−1 da un número mayor","Porque n es siempre par"],1,
    "Al calcular σ con el promedio muestral, se gasta un grado de libertad. Dividir por (n−1) evita subestimar la dispersión: es la corrección de Bessel.", id_preg="Err_p3_q_11", peso=2.0)
quiz_opcion_multiple("Si cuadruplicas el número de mediciones (de 4 a 16), ¿qué pasa con el error de la media?",
    ["Se reduce a 1/4","Se reduce a la mitad (porque σ_x̄ ∝ 1/√n)","No cambia","Se duplica"],1,
    "σ_x̄=σ/√n. Pasar de n=4 a n=16 multiplica √n por 2, así que el error de la media se reduce a la mitad, no a un cuarto.", id_preg="Err_p3_q_12", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: para reducir el error de la media a la CUARTA parte, un estudiante propone tomar 4 veces más mediciones. ¿Correcto o solo convincente?",
    ["Correcto: 4 veces más datos, 4 veces menos error","Solo convincente: como σ_x̄∝1/√n, para reducir el error a 1/4 se necesitan 16 veces más datos, no 4. Cuadruplicar n solo lo baja a la mitad","Correcto si los datos son buenos","No se puede reducir el error"],1,
    "El error de la media va como 1/√n. Para bajarlo a 1/4 hace falta √n=4, o sea n×16. La relación no es lineal: 'más datos = proporcionalmente menos error' es la trampa.", id_preg="Err_p3_concep_13", peso=2.0)


## Punto 4 — Propagación de errores  ★★★  [UCM, PRC]
Cuando una magnitud se **calcula** a partir de otras que medimos (con sus incertidumbres), el error se **propaga** al resultado. Reglas prácticas (para errores relativos que se suman):

- **Producto/división** (A = L·W): los errores relativos se suman → ΔA/A = ΔL/L + ΔW/W
- **Potencia** (A = πr²): el error relativo se multiplica por el exponente → ΔA/A = 2·(Δr/r)
- **Suma/resta:** se suman los errores absolutos

El resultado se reporta como valor ± incertidumbre, con cifras coherentes.

> 🔧 **Aplicación en ingeniería:** en integridad de tuberías (API/ASME), el área o el volumen se calculan a partir de mediciones de espesor y diámetro, cada una con su incertidumbre. Propagar correctamente el error determina si una tubería pasa o no el criterio de aceptación. Un error mal propagado puede aprobar una pieza defectuosa o rechazar una buena.

> 🤖 **TIP-IA — Verificador crítico:**
> Escribe: *"Propagué el error para calcular el área de un rectángulo y obtuve [tu ΔA]. Actúa como evaluador escéptico: revisa si sumé los errores RELATIVOS (no los absolutos) y si identifiqué bien cuál dimensión contribuye más. No me des la solución, solo dime dónde mirar."* 

In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\\d)\\.(\\d)", r"\\1,\\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

def mostrar_solucion(func_solucion):
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output(); func_solucion()
    boton.on_click(_click)
    display(boton, salida)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=opciones, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            ok = (radio.index==indice_correcto)
            print("\u2705 \u00a1Correcto!" if ok else "\u274c Incorrecto. Int\u00e9ntalo de nuevo.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.replace(",", "."))
            except ValueError: print("\u26a0 Escribe un n\u00famero."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            if ok: print(f"\u2705 \u00a1Correcto!  ({valor_correcto:.3g} {unidad})")
            else: print("\u274c Fuera de rango. Revisa tu procedimiento.")
            if explicacion: print("\u2192 " + explicacion)
            if id_preg is not None: _registrar(id_preg, ok, peso)
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not REGISTRO_BR:
                print("A\u00fan no has respondido ning\u00fan cuestionario. Responde y vuelve a intentar.")
                return
            peso_total = sum(p for _,p in REGISTRO_BR.values())
            peso_ok = sum(p for ok,p in REGISTRO_BR.values() if ok)
            nota = 5.0*peso_ok/peso_total if peso_total>0 else 0.0
            resp = len(REGISTRO_BR); aciertos = sum(1 for ok,_ in REGISTRO_BR.values() if ok)
            print("="*42)
            print("        CALIFICACI\u00d3N FORMATIVA")
            print("="*42)
            print(f"  Preguntas respondidas: {resp}" + (f" de {total_preguntas}" if total_preguntas else ""))
            print(f"  Respuestas correctas:  {aciertos}")
            print(f"  Puntaje ponderado:     {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*42)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*42)
            if nota>=3.5: print("  \u2705 \u00a1Buen trabajo! Dominas los conceptos.")
            elif nota>=3.0: print("  \U0001f7e1 Vas bien, repasa lo que fallaste.")
            else: print("  \U0001f534 Repasa la teor\u00eda y reintenta los puntos.")
            print("\n  Nota formativa (para tu pr\u00e1ctica). Puedes")
            print("  reintentar los cuestionarios y recalcular.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>\U0001f4ca Tu calificaci\u00f3n del taller</h3>"
        "<i>Responde los cuestionarios y pulsa el bot\u00f3n para ver tu nota ponderada por dificultad.</i>"),
        boton, salida]))

try:
    import matplotlib.pyplot as _ps
    _ps.rcParams['axes.spines.top']=False; _ps.rcParams['axes.spines.right']=False
    _ps.rcParams['axes.titlesize']=12; _ps.rcParams['axes.titleweight']='bold'
    _ps.rcParams['axes.titlecolor']='#2c3e50'; _ps.rcParams['axes.labelcolor']='#34495e'
    _ps.rcParams['axes.grid']=True; _ps.rcParams['grid.alpha']=0.25; _ps.rcParams['grid.linestyle']='--'
    _ps.rcParams['axes.facecolor']='#fcfcfd'
except Exception: pass

def esquema():
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(10,4))
    # IZQUIERDA: rectángulo con incertidumbre en L y W
    L,W,dL,dW=5,3,0.4,0.4
    ax1.add_patch(plt.Rectangle((0,0),L,W,fill=True,fc='#3498db',alpha=0.2,ec='#2980b9',lw=2))
    # banda de incertidumbre
    ax1.add_patch(plt.Rectangle((-dL/2,-dW/2),L+dL,W+dW,fill=False,ec='#c0392b',lw=1.5,ls='--'))
    ax1.annotate('',xy=(L,-0.6),xytext=(0,-0.6),arrowprops=dict(arrowstyle='<->',color='#2980b9'))
    ax1.text(L/2,-0.9,'L ± ΔL',ha='center',fontsize=9,color='#2980b9',fontweight='bold')
    ax1.annotate('',xy=(-0.6,W),xytext=(-0.6,0),arrowprops=dict(arrowstyle='<->',color='#e67e22'))
    ax1.text(-1.1,W/2,'W ± ΔW',ha='center',fontsize=9,color='#e67e22',fontweight='bold',rotation=90)
    ax1.text(L/2,W/2,'A = L·W\nΔA/A = ΔL/L + ΔW/W',ha='center',va='center',fontsize=9,color='#2c3e50',fontweight='bold')
    ax1.set_xlim(-1.5,6); ax1.set_ylim(-1.3,4); ax1.set_aspect('equal'); ax1.axis('off')
    ax1.set_title('Área rectangular: errores relativos se suman',fontsize=10)
    # DERECHA: cómo el exponente amplifica el error (potencia)
    exp=np.array([1,2,3]); err_r=0.02
    err_result=exp*err_r*100
    ax2.bar(['L¹\n(longitud)','r²\n(área)','r³\n(volumen)'],err_result,color=['#27ae60','#e67e22','#c0392b'],alpha=0.75)
    for i,e in enumerate(err_result):
        ax2.text(i,e+0.1,f'{e:.0f}%',ha='center',fontsize=10,fontweight='bold')
    ax2.set_ylabel('error relativo del resultado (%)')
    ax2.set_title('Un error del 2% en r se amplifica con el exponente',fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def propagacion(L=21.4, W=15.2, dL=0.1, dW=0.1):
    A=L*W
    dA=A*(dL/L+dW/W)
    eps=dA/A*100
    contrib_L=dL/L*100; contrib_W=dW/W*100
    fig,ax=plt.subplots(figsize=(8,3.8))
    ax.barh(['Contribución de L','Contribución de W'],[contrib_L,contrib_W],
            color=['#2980b9','#e67e22'],alpha=0.75)
    ax.text(contrib_L,0,f'  {contrib_L:.3g}%',va='center',fontsize=10,fontweight='bold')
    ax.text(contrib_W,1,f'  {contrib_W:.3g}%',va='center',fontsize=10,fontweight='bold')
    ax.set_xlabel('contribución al error relativo del área (%)')
    ax.set_title(f'A = ({A:.0f} ± {dA:.0f}) cm²  ·  ε = {eps:.2f}%')
    plt.tight_layout(); plt.show()

interact(propagacion,
    L=FloatSlider(value=21.4,min=10,max=30,step=1,description='L (cm)'),
    W=FloatSlider(value=15.2,min=5,max=25,step=1,description='W (cm)'),
    dL=FloatSlider(value=0.1,min=0.05,max=0.5,step=0.05,description='ΔL (cm)'),
    dW=FloatSlider(value=0.1,min=0.05,max=0.5,step=0.05,description='ΔW (cm)'));

def solucion():
    import re as _re
    def _pc(*a,**k):
        def _f(t):
            if not isinstance(t,str): return t
            return _re.sub(r'(\d)\.(\d)', r'\1,\2', t)
        print(*[_f(x) for x in a], **k)
    _pc("ÁREA RECTANGULAR A = L·W  (ejemplo L=21,4±0,1; W=15,2±0,1 cm):")
    L,W,dL,dW=21.4,15.2,0.1,0.1; A=L*W; dA=A*(dL/L+dW/W)
    _pc(f"   A = L·W = {A:.4g} cm²")
    _pc(f"   ΔA/A = ΔL/L + ΔW/W = {dL/L:.4g} + {dW/W:.4g} = {dA/A:.4g}")
    _pc(f"   ΔA = {dA:.3g} cm²  →  A = ({A:.0f} ± {dA:.0f}) cm², ε = {dA/A*100:.2f}%")
    _pc(f"   W contribuye más ({dW/W*100:.2f}%) que L ({dL/L*100:.2f}%): misma ΔX, pero W es menor,")
    _pc("   así que su error RELATIVO pesa más. Para mejorar, mide W con más cuidado.")
    _pc("")
    _pc("ÁREA CIRCULAR A = πr²  (ejemplo r=5,30±0,05 cm):")
    r,dr=5.30,0.05; Ac=np.pi*r**2; dAc=2*np.pi*r*dr
    _pc(f"   A = πr² = {Ac:.4g} cm²")
    _pc(f"   ΔA = 2πr·Δr = {dAc:.3g} cm²  (el exponente 2 DUPLICA el error relativo de r)")
    _pc(f"   A = ({Ac:.1f} ± {dAc:.1f}) cm², ε = {dAc/Ac*100:.2f}%")
    _pc("")
    _pc("COMPARACIÓN regla vs vernier (misma r=5,30 cm):")
    _pc(f"   Regla (Δr=0,05): ε = {2*np.pi*r*0.05/Ac*100:.3g}%")
    _pc(f"   Vernier (Δr=0,001): ε = {2*np.pi*r*0.001/Ac*100:.3g}%  → 50 veces mejor")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Rectángulo L=(21,4±0,1) cm, W=(15,2±0,1) cm. Calcule el área A=L·W (en cm²).",325,0.02,"cm²",
    "A=21,4×15,2=325,3≈325 cm².", id_preg="Err_p4_q_14", peso=3.0)
quiz_numerico("Para ese rectángulo, calcule la incertidumbre del área ΔA (en cm²).",3.66,0.08,"cm²",
    "ΔA=A(ΔL/L+ΔW/W)=325,3×(0,1/21,4+0,1/15,2)≈3,66 cm².", id_preg="Err_p4_q_15", peso=3.0)
quiz_numerico("Círculo con r=(5,30±0,05) cm. Calcule ΔA=2πr·Δr (en cm²).",1.67,0.05,"cm²",
    "ΔA=2π×5,30×0,05≈1,67 cm².", id_preg="Err_p4_q_16", peso=3.0)
quiz_opcion_multiple("En A=πr², ¿por qué un error del 2% en r produce un error del 4% en el área?",
    ["Por casualidad","Porque en una potencia rⁿ el error relativo se multiplica por el exponente n (aquí n=2)","Porque π amplifica el error","Porque el área es más grande"],1,
    "Para A=rⁿ, ΔA/A = n·(Δr/r). Con n=2: el error relativo del área es el doble del de r. Por eso 2% en r da 4% en A.", id_preg="Err_p4_q_17", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: en A=L·W con ΔL=ΔW (misma incertidumbre absoluta), un estudiante dice que L y W contribuyen igual al error. ¿Correcto?",
    ["Correcto: misma ΔX, misma contribución","Solo convincente: lo que se suma son los errores RELATIVOS (ΔL/L y ΔW/W). Si L≠W, la dimensión MENOR tiene mayor error relativo y contribuye más, aunque su ΔX sea igual","Correcto siempre","Depende del área total"],1,
    "En productos se suman errores relativos, no absolutos. Con la misma ΔX, la dimensión menor tiene mayor error relativo y pesa más. Confundir absoluto con relativo es la trampa.", id_preg="Err_p4_concep_18", peso=2.0)


---
## 📊 Tu calificación del taller
Pulsa el botón para calcular tu nota formativa (0,0 a 5,0), ponderada por la dificultad de cada punto. Puedes reintentar las veces que quieras.


In [ ]:
calificacion_final(total_preguntas=18)

---
### ✅ Fin del taller
Has trabajado la diferencia entre exactitud y precisión, los tipos de error y sus métricas, la estadística de mediciones repetidas (promedio, desviación estándar, error de la media) y la propagación de errores en magnitudes derivadas. Estos son los cimientos del trabajo experimental en ingeniería.

**Para practicar en casa:** mide objetos de tu hogar con una regla, repite cada medición tres veces, calcula el promedio y la incertidumbre, y propaga el error al calcular áreas. Usa monedas colombianas ($100≈5,50 g; $200≈4,50 g; $500≈7,10 g) como masas de referencia.

> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*
